In [4]:
# ============================================================
# RSNA KNEE ABNORMALITY DETECTION
# COMPLETE CNN ML SOLUTION
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ============================================================
# 1. SETTINGS
# ============================================================

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
TRAIN_SERIES_CSV = os.path.join(DATA_ROOT, "train_series.csv")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
TEST_SERIES_CSV = os.path.join(DATA_ROOT, "test_series.csv")

TRAIN_DIR = os.path.join(DATA_ROOT, "train_series")
TEST_DIR = os.path.join(DATA_ROOT, "test_series")

LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 0.0005

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Starting RSNA Knee Abnormality Detection")
print("Device:", device)
print("Data path:", DATA_ROOT)


# ============================================================
# 2. LOAD CSV FILES
# ============================================================

train = pd.read_csv(TRAIN_CSV)
train_series = pd.read_csv(TRAIN_SERIES_CSV)
test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)

print("\nTRAIN:", train.shape)
print("TRAIN SERIES:", train_series.shape)
print("TEST:", test.shape)
print("TEST SERIES:", test_series.shape)


# ============================================================
# 3. PREPARE LABELS
# ============================================================

# Use official labels where available.
# Missing labels are temporarily filled using simple
# report-based keyword detection.

patterns = {

    "ACL": [
        r"acl.*tear",
        r"acl.*rupture",
        r"anterior cruciate.*tear",
        r"anterior cruciate.*rupture"
    ],

    "MCL": [
        r"mcl.*tear",
        r"mcl.*rupture",
        r"medial collateral.*tear",
        r"medial collateral.*rupture"
    ],

    "Medial Meniscus": [
        r"medial meniscus.*tear",
        r"medial meniscus.*rupture",
        r"meniscus.*medial.*tear",
        r"meniscal tear"
    ],

    "Lateral Meniscus": [
        r"lateral meniscus.*tear",
        r"lateral meniscus.*rupture",
        r"meniscus.*lateral.*tear"
    ],

    "Medial OA": [
        r"medial.*osteoarthritis",
        r"medial.*arthrosis",
        r"medial.*degenerative"
    ],

    "Lateral OA": [
        r"lateral.*osteoarthritis",
        r"lateral.*arthrosis",
        r"lateral.*degenerative"
    ],

    "PF OA": [
        r"patellofemoral.*osteoarthritis",
        r"patellofemoral.*arthrosis",
        r"chondromalacia patella"
    ],

    "Effusion": [
        r"joint effusion",
        r"effusion",
        r"derrame"
    ],

    "Synovitis": [
        r"synovitis",
        r"synovial.*thickening"
    ],

    "Baker's": [
        r"baker.*cyst",
        r"popliteal cyst"
    ],

    "Contusion": [
        r"bone contusion",
        r"bone bruise",
        r"contusion",
        r"bone marrow edema"
    ],

    "Fracture": [
        r"fracture",
        r"fractured"
    ]
}


def get_weak_label(report, label):

    if not isinstance(report, str):
        return 0

    text = report.lower()

    for pattern in patterns[label]:

        match = re.search(pattern, text)

        if match:

            context = text[
                max(0, match.start() - 80):
                min(len(text), match.end() + 30)
            ]

            negations = [
                "no ",
                "without",
                "normal",
                "intact",
                "no evidence",
                "sin signos",
                "sin evidencia",
                "aucune",
                "kein",
                "keine"
            ]

            if any(n in context for n in negations):
                continue

            return 1

    return 0


weak_labels = pd.DataFrame(
    index=train.index
)

for label in LABELS:

    weak_labels[label] = train["Report"].apply(
        lambda x: get_weak_label(x, label)
    )


# Official labels override weak labels

final_labels = weak_labels.copy()

for label in LABELS:

    if label in train.columns:

        mask = train[label].notna()

        final_labels.loc[mask, label] = (
            train.loc[mask, label]
        )


# Make StudyInstanceUID the index

train["StudyInstanceUID"] = (
    train["StudyInstanceUID"].astype(str)
)

final_labels.index = train["StudyInstanceUID"]

print("\nLabel matrix:", final_labels.shape)


# ============================================================
# 4. SELECT ONE USEFUL MRI SERIES PER STUDY
# ============================================================

def choose_series(group):

    # First prefer fluid-sensitive + fat-suppressed
    preferred = group[
        (group["Fluid_Sensitive"] == 1) &
        (group["Fat_Suppression"] == 1)
    ]

    if len(preferred) > 0:
        group = preferred

    # Then prefer sagittal
    sagittal = group[
        group["Anatomical_Plane"]
        .astype(str)
        .str.lower()
        == "sagittal"
    ]

    if len(sagittal) > 0:
        return sagittal.iloc[0]

    return group.iloc[0]


selected_train_series = []

for study_id, group in train_series.groupby(
    "StudyInstanceUID"
):

    selected_train_series.append(
        choose_series(group)
    )

selected_train_series = pd.DataFrame(
    selected_train_series
).reset_index(drop=True)


selected_test_series = []

for study_id, group in test_series.groupby(
    "StudyInstanceUID"
):

    selected_test_series.append(
        choose_series(group)
    )

selected_test_series = pd.DataFrame(
    selected_test_series
).reset_index(drop=True)


print("\nTraining studies:", len(train))
print(
    "Selected training series:",
    len(selected_train_series)
)

print(
    "Test studies:",
    len(test)
)

print(
    "Selected test series:",
    len(selected_test_series)
)


# ============================================================
# 5. TRAIN / VALIDATION SPLIT
# ============================================================

study_ids = selected_train_series[
    "StudyInstanceUID"
].astype(str).unique()

train_ids, val_ids = train_test_split(
    study_ids,
    test_size=0.15,
    random_state=42
)

print("\nTraining split:", len(train_ids))
print("Validation split:", len(val_ids))


# ============================================================
# 6. DICOM DATASET
# ============================================================

class KneeDataset(Dataset):

    def __init__(
        self,
        series_dataframe,
        labels_dataframe,
        data_root
    ):

        self.series_dataframe = (
            series_dataframe.reset_index(drop=True)
        )

        self.labels_dataframe = labels_dataframe

        self.data_root = data_root


    def __len__(self):

        return len(self.series_dataframe)


    def load_middle_slice(
        self,
        study_id,
        series_id
    ):

        series_path = os.path.join(
            self.data_root,
            str(study_id),
            str(series_id)
        )

        if not os.path.exists(series_path):

            return np.zeros(
                (IMG_SIZE, IMG_SIZE),
                dtype=np.float32
            )


        files = glob.glob(
            os.path.join(series_path, "*.dcm")
        )


        if len(files) == 0:

            return np.zeros(
                (IMG_SIZE, IMG_SIZE),
                dtype=np.float32
            )


        slices = []


        for file in files:

            try:

                ds = pydicom.dcmread(
                    file,
                    force=True
                )

                if not hasattr(ds, "pixel_array"):
                    continue

                image = ds.pixel_array.astype(
                    np.float32
                )


                # Apply DICOM scaling if available

                if hasattr(ds, "RescaleSlope"):

                    image *= float(
                        ds.RescaleSlope
                    )

                if hasattr(ds, "RescaleIntercept"):

                    image += float(
                        ds.RescaleIntercept
                    )


                # Sort using Z position

                position = 0

                if hasattr(
                    ds,
                    "ImagePositionPatient"
                ):

                    position = float(
                        ds.ImagePositionPatient[2]
                    )


                slices.append(
                    (position, image)
                )

            except Exception:

                continue


        if len(slices) == 0:

            return np.zeros(
                (IMG_SIZE, IMG_SIZE),
                dtype=np.float32
            )


        slices.sort(
            key=lambda x: x[0]
        )


        # Middle MRI slice

        middle_index = len(slices) // 2

        image = slices[middle_index][1]


        # Normalize

        image = image - image.min()

        if image.max() > 0:

            image = image / image.max()


        # Convert to tensor

        image = torch.tensor(
            image,
            dtype=torch.float32
        )


        # Resize

        image = torch.nn.functional.interpolate(
            image.unsqueeze(0).unsqueeze(0),
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear",
            align_corners=False
        )


        image = image.squeeze(0)

        return image


    def __getitem__(self, index):

        row = self.series_dataframe.iloc[index]

        study_id = str(
            row["StudyInstanceUID"]
        )

        series_id = str(
            row["SeriesInstanceUID"]
        )


        image = self.load_middle_slice(
            study_id,
            series_id
        )


        if self.labels_dataframe is not None:

            target = self.labels_dataframe.loc[
                study_id,
                LABELS
            ].values.astype(
                np.float32
            )

            target = torch.tensor(
                target,
                dtype=torch.float32
            )

            return image, target


        return image, study_id


# ============================================================
# 7. CREATE TRAIN / VALIDATION DATA
# ============================================================

selected_train_series[
    "StudyInstanceUID"
] = selected_train_series[
    "StudyInstanceUID"
].astype(str)


train_series_selected = selected_train_series[
    selected_train_series[
        "StudyInstanceUID"
    ].isin(train_ids)
].reset_index(drop=True)


val_series_selected = selected_train_series[
    selected_train_series[
        "StudyInstanceUID"
    ].isin(val_ids)
].reset_index(drop=True)


train_dataset = KneeDataset(
    train_series_selected,
    final_labels,
    TRAIN_DIR
)


val_dataset = KneeDataset(
    val_series_selected,
    final_labels,
    TRAIN_DIR
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print("\nDataLoaders created.")
print(
    "Training samples:",
    len(train_dataset)
)
print(
    "Validation samples:",
    len(val_dataset)
)


# ============================================================
# 8. IMPROVED CNN MODEL
# ============================================================

class ImprovedKneeCNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()


        self.features = nn.Sequential(

            # Block 1

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2),


            # Block 2

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(2),


            # Block 3

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            nn.MaxPool2d(2),


            # Block 4

            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(),


            nn.AdaptiveAvgPool2d(
                (1, 1)
            )
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.4),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


model = ImprovedKneeCNN(
    len(LABELS)
).to(device)


criterion = nn.BCEWithLogitsLoss()


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0001
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=1,
    factor=0.5
)


print("\nModel ready!")
print(
    "Output classes:",
    len(LABELS)
)


# ============================================================
# 9. TRAINING
# ============================================================

print("\n==============================")
print("TRAINING MODEL")
print("==============================")


best_val_loss = float("inf")


for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0.0


    for batch_number, (
        images,
        targets
    ) in enumerate(train_loader):


        images = images.to(device)

        targets = targets.to(device)


        optimizer.zero_grad()


        outputs = model(images)


        loss = criterion(
            outputs,
            targets
        )


        loss.backward()


        optimizer.step()


        total_train_loss += loss.item()


        if (
            (batch_number + 1) % 20 == 0
        ):

            print(
                "Epoch",
                epoch + 1,
                "| Batch",
                batch_number + 1,
                "/",
                len(train_loader)
            )


    train_loss = (
        total_train_loss /
        len(train_loader)
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model.eval()

    total_val_loss = 0.0


    with torch.no_grad():

        for images, targets in val_loader:

            images = images.to(device)

            targets = targets.to(device)


            outputs = model(images)


            loss = criterion(
                outputs,
                targets
            )


            total_val_loss += loss.item()


    val_loss = (
        total_val_loss /
        len(val_loader)
    )


    scheduler.step(val_loss)


    print()
    print(
        "Epoch",
        epoch + 1,
        "complete"
    )

    print(
        f"Training Loss: {train_loss:.4f}"
    )

    print(
        f"Validation Loss: {val_loss:.4f}"
    )

    print(
        "Learning Rate:",
        optimizer.param_groups[0]["lr"]
    )

    print(
        "------------------------------"
    )


    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_knee_model.pth"
        )


# ============================================================
# 10. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        "/kaggle/working/best_knee_model.pth",
        map_location=device
    )
)

model.eval()

print("\nBest validation loss:", best_val_loss)


# ============================================================
# 11. TEST PREDICTIONS
# ============================================================

selected_test_series[
    "StudyInstanceUID"
] = selected_test_series[
    "StudyInstanceUID"
].astype(str)


all_predictions = []

all_study_ids = []


print("\nGenerating test predictions...")


with torch.no_grad():

    for _, row in selected_test_series.iterrows():

        study_id = str(
            row["StudyInstanceUID"]
        )

        series_id = str(
            row["SeriesInstanceUID"]
        )


        image = load_image = None

        # Load MRI image using the same dataset logic

        image = KneeDataset(
            pd.DataFrame([row]),
            None,
            TEST_DIR
        ).load_middle_slice(
            study_id,
            series_id
        )


        image = torch.tensor(
            image,
            dtype=torch.float32
        ).unsqueeze(0).to(device)


        outputs = model(image)


        probabilities = torch.sigmoid(
            outputs
        ).cpu().numpy()[0]


        all_predictions.append(
            probabilities
        )

        all_study_ids.append(
            study_id
        )


predictions = np.array(
    all_predictions
)


print(
    "Prediction shape:",
    predictions.shape
)


# ============================================================
# 12. CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame(
    predictions,
    columns=LABELS
)


submission.insert(
    0,
    "StudyInstanceUID",
    all_study_ids
)


submission_path = (
    "/kaggle/working/submission.csv"
)


submission.to_csv(
    submission_path,
    index=False
)


print("\n==============================")
print("SUBMISSION CREATED")
print("==============================")

print(
    "Shape:",
    submission.shape
)

print(
    "File:",
    submission_path
)


display(submission)


# ============================================================
# 13. DETECTION RESULTS
# ============================================================

threshold = 0.5


detection_results = submission.copy()


for label in LABELS:

    detection_results[label] = (
        detection_results[label] >= threshold
    ).map({
        True: "Detected",
        False: "Not Detected"
    })


print("\n==============================")
print("KNEE ABNORMALITY DETECTION")
print("==============================")


display(detection_results)


# ============================================================
# 14. SAVE DETECTION RESULTS
# ============================================================

detection_results.to_csv(
    "/kaggle/working/detection_results.csv",
    index=False
)


print("\nFiles created:")

print(
    "/kaggle/working/submission.csv"
)

print(
    "/kaggle/working/detection_results.csv"
)

print(
    "/kaggle/working/best_knee_model.pth"
)

print("\nDONE!")

Starting RSNA Knee Abnormality Detection
Device: cpu
Data path: /kaggle/input/competitions/rsna-knee-abnormality-detection

TRAIN: (4407, 14)
TRAIN SERIES: (24371, 5)
TEST: (3, 1)
TEST SERIES: (15, 5)

Label matrix: (4407, 12)

Training studies: 4407
Selected training series: 4407
Test studies: 3
Selected test series: 3

Training split: 3745
Validation split: 662

DataLoaders created.
Training samples: 3745
Validation samples: 662

Model ready!
Output classes: 12

TRAINING MODEL
Epoch 1 | Batch 20 / 235
Epoch 1 | Batch 40 / 235
Epoch 1 | Batch 60 / 235
Epoch 1 | Batch 80 / 235
Epoch 1 | Batch 100 / 235
Epoch 1 | Batch 120 / 235
Epoch 1 | Batch 140 / 235
Epoch 1 | Batch 160 / 235
Epoch 1 | Batch 180 / 235
Epoch 1 | Batch 200 / 235
Epoch 1 | Batch 220 / 235

Epoch 1 complete
Training Loss: 0.1998
Validation Loss: 0.1531
Learning Rate: 0.0005
------------------------------
Epoch 2 | Batch 20 / 235
Epoch 2 | Batch 40 / 235
Epoch 2 | Batch 60 / 235
Epoch 2 | Batch 80 / 235
Epoch 2 | Batch 1

/tmp/ipykernel_58/1970753532.py:954: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  image = torch.tensor(


Prediction shape: (3, 12)

SUBMISSION CREATED
Shape: (3, 13)
File: /kaggle/working/submission.csv


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.022397,0.008106,0.040190,0.017169,0.018946,0.009377,0.012319,0.101715,0.038710,0.032695,0.059349,0.017150
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.057631,0.028210,0.095999,0.049238,0.059117,0.030217,0.038911,0.215672,0.098429,0.071703,0.130759,0.047582
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.010387,0.003069,0.021569,0.007696,0.008961,0.002859,0.004653,0.099096,0.017468,0.012427,0.031316,0.005447



KNEE ABNORMALITY DETECTION


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected,Not Detected



Files created:
/kaggle/working/submission.csv
/kaggle/working/detection_results.csv
/kaggle/working/best_knee_model.pth

DONE!
